# Train Graph Transformer for Redundancy Prediction
This notebook mounts Google Drive, sets up the environment, and trains the PyTorch Geometric Graph Transformer on the `training_pairs.csv` dataset.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

# ==============================================================================
# Step 1: Mount Drive and set environment variable
# ==============================================================================
print("[STEP 1] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])


In [ ]:
# ==============================================================================
# Step 2: Verify GPU
# ==============================================================================
print("\n[STEP 2] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# Step 3: Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

print("\n[INFO] Installing PyTorch Geometric...")
!pip install -q torch-geometric


In [ ]:
# ==============================================================================
# Step 4: Run Training
# ==============================================================================
print("\n[STEP 4] Running Graph Transformer Training...")
!python -m src.train_graph_transformer


In [ ]:
# ==============================================================================
# Step 5: Verify Saved Model
# ==============================================================================
# NOTE: The training script saves the model to 'models/' in the repo, NOT drive root directly.
# Let's copy it to Drive so it persists after Colab shuts down!
print("\n[STEP 5] Persisting Model to Google Drive...")
local_model = Path("models/graph_transformer.pt")
drive_model = DRIVE_ROOT / "models" / "graph_transformer.pt"
drive_model.parent.mkdir(parents=True, exist_ok=True)

if local_model.exists():
    import shutil
    shutil.copy2(local_model, drive_model)
    size_mb = drive_model.stat().st_size / (1024 * 1024)
    print(f"[PASS] Model successfully copied to Drive: {drive_model} ({size_mb:.2f} MB)")
else:
    print("[FAIL] Model file was not generated by the training script.")
